In [42]:
# Import dependencies
import pandas as pd
import  pandas_datareader.data as web
from datetime import datetime
import yfinance as yf
import matplotlib.pyplot as plt
import fredapi
import seaborn as sns

### Money at a fixed rate for an secured purchase: 

Datasets to obtain: 
1. SOFR rate
2. Federal Funds rate

#### Fetch the data
1. SOFR Data

In [ ]:
#import the SOFR data: 

# Define the date range
start = datetime(2019, 1, 1)
end = datetime(2023, 12, 31)

# Fetch SOFR data
SOFR_df = web.DataReader('SOFR', 'fred', start, end)
SOFR_df = SOFR_df.dropna()

# Resampling to monthly data
SOFR_df = SOFR_df.resample('M').mean().reset_index()

SOFR_df

2. Federal Funds rate

In [ ]:
# Importing the federal funds rate
import pandas_datareader.data as web
from datetime import datetime

# Define the date range
start = datetime(2019, 1, 1)
end = datetime(2023, 12, 31)

# Retrieve the federal funds rate data using pandas-datareader
fed_funds_rate = web.get_data_fred('FEDFUNDS', start=start, end=end)
fed_funds_rate = fed_funds_rate.dropna()

fed_funds_rate = fed_funds_rate.resample('M').mean().reset_index()

# Print the data
fed_funds_rate

In [ ]:
# Get VIX data
vix = yf.Ticker('^VIX')
vix_data = vix.history(start='2023-01-01', end='2023-12-31')
vix_data

#### Combining the dataframe

In [ ]:
#Combine the dataframes
secured_combined_df = fed_funds_rate.copy()
secured_combined_df.rename(columns={'FEDFUNDS': 'Fed Funds Rate', 'DATE': 'Date'}, inplace=True)
secured_combined_df["SOFR"] = SOFR_df['SOFR'].round(2)
secured_combined_df

#### Analysing the data

In [ ]:
secured_combined_df[['Fed Funds Rate', 'SOFR']].describe()

#### 1) SOFR rates

In [ ]:
# Visualizing the SOFR over time
plt.figure(figsize=(12, 6))
plt.plot(SOFR_df['DATE'], SOFR_df['SOFR'])
plt.xlabel('Date')
plt.ylabel('SOFR Rate')
plt.title('SOFR Rate Over Time')
plt.show()

The above graph shows the SOFR rate from 2018. We can see that the rate was relatively low during 2018 and 2019 and continued to increase till 2019. After which see saw a consistent and exponential decline till the start of 2020.
After which it was consistently extremly low till the beginning of 2022. After which we saw a rapid exponential increase in the rates. Leading to values that are higher than ever before.

In terms of leverage, we can see that the rate over the past 5 years has a standard deviation of 1.91, which shows that there is a huge spread in the rates. Which increases the leverage opportunities.

#### 2) Federal Funds Rate

In [ ]:
# Visualizing the SOFR over time
plt.figure(figsize=(12, 6))
plt.plot(secured_combined_df['Date'], secured_combined_df['Fed Funds Rate'])
plt.xlabel('Date')
plt.ylabel('Funds rate')
plt.title('Federal Funds rate Over Time')
plt.show()

In [ ]:
correlation = secured_combined_df['Fed Funds Rate'].corr(secured_combined_df['SOFR'])
correlation

### Publically traded equity: 

#### Datasets to obtain: 
1. Stock data (Netflix)
2. Total Secured borrowing of Hedge Funds from FRED (Margin requirements)

#### 1. Stock data

In [ ]:
#Get Netflix price data from the yfinance

# Download historical data of Netflix
netflix_df = pd.DataFrame(yf.download("NFLX", start='2016-01-01', end='2023-12-31')).reset_index()
netflix_df = netflix_df[["Date", "Close"]]
netflix_df = netflix_df.groupby(pd.Grouper(key='Date', freq='M'))['Close'].mean()
netflix_df.reset_index()

# Resample to Monthly and calculate the stock prices
netflix_df = netflix_df.resample('Q').mean().reset_index()
netflix_df

#### 2. Margin Debt (Hedge fund margin requirements)

In [ ]:
fred = fredapi.Fred(api_key="0d71ce218db7ed181282195166426c5a")
series_id = "BOGZ1FL624123035Q"
data = fred.get_series(series_id,)

margin_df = pd.DataFrame(data, columns=["Value"])
margin_df.index.name = "Date"

# Data from 2016
margin_df = margin_df[margin_df.index >= '2016-01-01']
margin_df = margin_df.reset_index()

margin_df

In [ ]:
# Combine the dataframes
equity_df = margin_df.copy()
equity_df['NFLX'] = netflix_df['Close']
equity_df.rename(columns={"Value": "Margin",}, inplace=True)

equity_df

#### Analyse the data:

In [ ]:
equity_df[['NFLX', 'Margin']].describe()

#### 1) Netflix stock:

In [ ]:
# Visualizing the price of netflix over time
plt.figure(figsize=(12, 6))
plt.plot(equity_df['Date'], equity_df['NFLX'])
plt.xlabel('Date')
plt.ylabel('Netflix price')
plt.title('Netflix stock price change over time')
plt.show()

#### 2) Margin requirments (Hedge Fund)

In [ ]:
# Visualizing the price of netflix over time
plt.figure(figsize=(12, 6))
plt.plot(equity_df['Date'], equity_df['Margin'])
plt.xlabel('Date')
plt.ylabel('Margin')
plt.title('Margin requirements over time')
plt.show()

#### 3) Correlation analysis of Netflix and Margin requirement

In [ ]:
# Calculating corr matrix
corr_matrix2 = equity_df[['Margin', 'NFLX']].corr()

# Set the size of the plot
plt.figure(figsize=(8, 6))

# Create a heatmap
sns.heatmap(corr_matrix2, annot=True, cmap='coolwarm', fmt=".2f", square=True)

# Title and show the plot
plt.title('Correlation Heatmap')
plt.show()

print(equity_df[['NFLX', 'Margin']].corr())